In [1]:
from PIL import Image
import pytesseract
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
from langchain.tools import tool

@tool
def ocr_read_document(image_path:str)->str:
    """Reads image from given path and returns the text"""
    try:
        text=pytesseract.image_to_string(Image.open(image_path))
        return text
    except Exception as e:
        return f"Erro rading image :{e}"

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint



llm=HuggingFaceEndpoint(
    # repo_id="HuggingFaceH4/zephyr-7b-beta",
    # repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    # repo_id="zai-org/GLM-4.7-Flash",
    repo_id="openai/gpt-oss-20b",
    # repo_id="openai/gpt-oss-120b",
    # repo_id="HuggingFaceH4/zephyr-7b-gemma-v0.1",
    # repo_id="lmsys/vicuna-13b-v1.5",
    task="text-generation"
)
model=ChatHuggingFace(llm=llm)
# model=ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [5]:
print(model.invoke("Hello").content)

Hello! 👋 How can I help you today?


In [6]:
llm_with_tools=model.bind_tools([ocr_read_document])

In [19]:
from langchain_core.messages import HumanMessage,ToolMessage
messages=[HumanMessage(content="extract text from this image ,path:/home/abrar/Desktop/Abrar/LangGraph/LandingAI/img_1.png, and return the analysed results in json")]

In [20]:
result=llm_with_tools.invoke(messages)

In [23]:
messages.append(result)

In [24]:
for tool_call in result.tool_calls:
    if tool_call['name']=='ocr_read_document':
        response=ocr_read_document.invoke(tool_call['args'])
       
        messages.append(
            ToolMessage(
                content=str(response) ,
                tool_call_id=tool_call['id']
            )
        )

In [25]:
print(llm_with_tools.invoke(messages).content)

```json
{
  "shop_name": "SHOP NAME",
  "address": "Lorem Ipsum, 23-10",
  "phone": "11223344",
  "receipt_type": "CASH RECEIPT",
  "items": [
    { "description": "Lorem", "price": "1" },
    { "description": "Ipsum", "price": "22" },
    { "description": "Dolor sit amet", "price": "33" },
    { "description": "Consectetur", "price": "4a" },
    { "description": "Adipiscing elit", "price": "55" }
  ],
  "totals": {
    "total": "16.5",
    "cash": "20.0",
    "change": "35",
    "bank_card": "234",
    "approval_code": "#123456"
  },
  "thank_you": true,
  "raw_text": "SHOP NAME\nAddress: Lorem Ipsum, 23-10\nTelp. 11223344\n\nCASH RECEIPT\n\nDescription Price\nLorem 1\nIpsum 22\nDolor sit amet 33\nConsectetur 4a\nAdipiscing elit
